# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 7: Comparativo Multimodelo com o Extrator MEL (MLP, CNN, SVM, Random Forest e BiGRU)

Neste notebook comparamos as famílias de aprendizado de máquina avaliadas no projeto com o **log-MEL** (*log-melgram*): energia em `n_mels = 256` bandas na escala Mel, em escala logarítmica, que comprime a resolução nas altas frequências e preserva a envoltória espectral de banda larga (detalhes no Notebook 5). É o mesmo formato dos notebooks 9 (DEMON) e 11 (HHT), com o MEL no lugar do extrator.

| Modelo | Experimento | Grade temporal | Origem |
| :--- | :--- | :---: | :--- |
| MLP (≈ artigo) | `mlp` | 0,512 s | pipeline iara, `test_scripts/mlp.py`: arquitetura do artigo ([32,16], dropout 0,2, BatchNorm, ReLU, Sigmoid, lr 1e-4), mas até 512 épocas com paciência 16 (grid search do artigo: 50 épocas, paciência 8) |
| CNN (artigo) | `cnn_mel_artigo` | 0,512 s | pipeline iara, `test_scripts/cnn_mel_artigo.py`: configuração do grid search do artigo (conv 1024/128, LeakyReLU, dropout 0,4, lr 1e-6, imagens de 32 janelas com 50% de sobreposição, paciência 32) |
| CNN (repo) | `cnn` | 0,512 s | pipeline iara, `test_scripts/cnn.py`: conv 128/64, ReLU, lr 1e-5, imagens sem sobreposição — não é a arquitetura do artigo |
| SVM Nyström m=4000 | `svm_nystroem_4000_log_melgram_elasticnet_l1r0.15_C2.0` | 0,512 s | pipeline iara (`test_scripts/svm.py`), C=2,0, ElasticNet l1_ratio=0,15 |
| Random Forest 100 árv. | `rf_mel5s_n100_balanced_4c` | 5,12 s | `scratch/generate_rf_predictions_and_analysis.py`, média de 10 quadros de 0,512 s |
| MLP [32,16] 35 ép. | `mlp_mel5s_h32-16_e35_4c` | 5,12 s | `scratch/eval_fusion_mel_demon_5x2cv.py` (linha de base só MEL da fusão do Notebook 12) |
| BiGRU + atenção | `bigru_att_mel5s_h64_s15-5_e20_4c` | 5,12 s | `scratch/eval_bigru_multiclass_5x2cv.py`, sequências de 15 vetores |

Todos os modelos foram avaliados no protocolo oficial de **$5\times2$ validação cruzada (10 folds)** com **particionamento por gravação de áudio (`ID`, `exclusive_header='ID'` em `src/iara/ml/experiment.py`)** — janelas de um mesmo arquivo nunca ficam em treino e teste ao mesmo tempo — sobre os **1.706 áudios de UO** do IARA (*Silva et al.*, IEEE Access 2025). O particionamento **não** isola navios: gravações diferentes do mesmo navio (`Ship ID`) podem cair em treino e teste.

> **Duas grades temporais.** Os modelos do pipeline oficial (MLP, CNN e SVM) classificam janelas de **0,512 s**; os modelos ad hoc de `scratch/` (RF, CNN 2D, MLP 5,12 s, BiGRU) usam o cache de **5,12 s** (média de 10 quadros), o mesmo dos notebooks DEMON e HHT. A predição por arquivo é sempre o voto majoritário das janelas, mas modelos de grades diferentes não diferem só no classificador.

> Todos os números são recalculados das predições por janela em `results/trainings/` por `notebooks/src/iara_results.py` (voto majoritário por arquivo, SP de `src/iara/ml/metrics.py`, **ACC = acurácia balanceada**, média ± desvio populacional nos 10 folds).

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

for _p in ('.', 'src', 'notebooks', 'notebooks/src'):
    if os.path.exists(os.path.join(_p, 'iara_results.py')) and os.path.abspath(_p) not in sys.path:
        sys.path.insert(0, os.path.abspath(_p))
import iara_results as R

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 13,
                     'xtick.labelsize': 10, 'ytick.labelsize': 10, 'figure.titlesize': 15, 'figure.dpi': 120})

## 1. Tabela Mestra: MEL no Artigo e nas Re-execuções Locais

> **Notas metodológicas:**
> * **ACC = acurácia balanceada** (média dos recalls) em todas as tabelas, inclusive na Tabela 7 do artigo.
> * **Três modos, sempre rotulados:** `4c` (4 classes); `4c_sem_bg` (o mesmo modelo 4c avaliado só em navios, prever BACKGROUND = erro; os recalls por classe são os mesmos do 4c); `3c` (modelo **retreinado** sem BACKGROUND — só existe para MLP [32,16] 35 ép., BiGRU + atenção). Comparar `3c` com `4c_sem_bg` mistura problemas diferentes.
> * O artigo publica só SP, ACC e F1 médios (sem predições por fold); por isso não há F-test entre as re-execuções locais e o artigo.

In [ ]:
# ---- Experimentos em disco (results/trainings/{tests,multirep}); _3c = modelo RETREINADO sem BACKGROUND ----
REP = 'MEL'
OUTRA = 'LOFAR'
PRINCIPAIS = [
    ('MLP (≈ artigo)', 'mlp', None, '0,512 s', 'pipeline iara, `test_scripts/mlp.py`: arquitetura do artigo ([32,16], dropout 0,2, BatchNorm, ReLU, Sigmoid, lr 1e-4), mas até 512 épocas com paciência 16 (grid search do artigo: 50 épocas, paciência 8)'),
    ('CNN (artigo)', 'cnn_mel_artigo', None, '0,512 s', 'pipeline iara, `test_scripts/cnn_mel_artigo.py`: configuração do grid search do artigo (conv 1024/128, LeakyReLU, dropout 0,4, lr 1e-6, imagens de 32 janelas com 50% de sobreposição, paciência 32)'),
    ('CNN (repo)', 'cnn', None, '0,512 s', 'pipeline iara, `test_scripts/cnn.py`: conv 128/64, ReLU, lr 1e-5, imagens sem sobreposição — não é a arquitetura do artigo'),
    ('SVM Nyström m=4000', 'svm_nystroem_4000_log_melgram_elasticnet_l1r0.15_C2.0', None, '0,512 s', 'pipeline iara (`test_scripts/svm.py`), C=2,0, ElasticNet l1_ratio=0,15'),
    ('Random Forest 100 árv.', 'rf_mel5s_n100_balanced_4c', None, '5,12 s', '`scratch/generate_rf_predictions_and_analysis.py`, média de 10 quadros de 0,512 s'),
    ('MLP [32,16] 35 ép.', 'mlp_mel5s_h32-16_e35_4c', 'mlp_mel5s_h32-16_e35_3c', '5,12 s', '`scratch/eval_fusion_mel_demon_5x2cv.py` (linha de base só MEL da fusão do Notebook 12)'),
    ('BiGRU + atenção', 'bigru_att_mel5s_h64_s15-5_e20_4c', 'bigru_att_mel5s_h64_s15-5_e20_3c', '5,12 s', '`scratch/eval_bigru_multiclass_5x2cv.py`, sequências de 15 vetores'),
]  # (rótulo, exp 4c, exp 3c ou None, grade temporal, origem)
SVM_SWEEP = [
    ('m=300', 'svm_nystroem'),
    ('m=1000', 'svm_nystroem_1000'),
    ('m=1000, norm, PCA64', 'svm_nystroem_1000_log_melgram_norm_pca64'),
    ('m=2000', 'svm_nystroem_2000_log_melgram'),
    ('m=2000, C=10', 'svm_nystroem_2000_log_melgram_C10.0'),
    ('m=3000', 'svm_nystroem_3000_log_melgram'),
    ('m=4000, C=0.5, ElasticNet', 'svm_nystroem_4000_log_melgram_elasticnet_l1r0.15_C0.5'),
    ('m=4000, C=2.0, ElasticNet', 'svm_nystroem_4000_log_melgram_elasticnet_l1r0.15_C2.0'),
    ('m=4000, C=2.0, PCA64, ElasticNet', 'svm_nystroem_4000_log_melgram_pca64_elasticnet_l1r0.15_C2.0'),
    ('m=4000, landmarks k-means, ElasticNet', 'svm_nystroem_4000_log_melgram_kmeans_elasticnet_l1r0.15'),
]
PARES_OUTRA = [
    ('MLP (≈ artigo, 0,512 s)', 'mlp', 'mlp_lofar'),
    ('SVM Nyström (melhor de cada)', 'svm_nystroem_4000_log_melgram_elasticnet_l1r0.15_C2.0', 'svm_nystroem_6000_lofar_elasticnet_l1r0.15'),
    ('Random Forest (5,12 s)', 'rf_mel5s_n100_balanced_4c', 'rf_lofar5s_n100_balanced_4c'),
    ('BiGRU + atenção (5,12 s)', 'bigru_att_mel5s_h64_s15-5_e20_4c', 'bigru_att_lofar5s_h128_s15-5_e15_4c'),
]  # (arquitetura, exp com REP, exp com OUTRA)
MET = ('SP', 'ACC', 'F1')

E4 = {l: e for l, e, *_ in PRINCIPAIS}
E3 = {l: e3 for l, _, e3, *_ in PRINCIPAIS if e3}
S4 = {l: R.summary(e, '4c') for l, e in E4.items()}
SB = {l: R.summary(e, '4c_sem_bg') for l, e in E4.items()}
S3 = {l: R.summary(e, '3c') for l, e in E3.items()}
FOLDS = {l: R.fold_metrics(e) for l, e in E4.items()}
# Colapso: em algum fold, alguma classe com recall < 5% (o SP do fold despenca e a média deixa de ser informativa)
COLAPSADO = {l: bool((FOLDS[l][[f'R_{c}' for c in R.CLASSES]].min(axis=1) < 5).any()) for l in E4}

# Tabela 7 do artigo (Silva et al., IEEE Access 2025): só SP, ACC e F1 em 4 classes
tab_artigo = pd.DataFrame([{'Modelo': f'{m} | {r} (Tabela 7 do artigo)',
                            **{k: R.fmt(v) for k, v in R.ARTICLE_TABLE7[(m, r)].items()}}
                           for (m, r) in R.ARTICLE_TABLE7 if r == REP]).set_index('Modelo')
ROWS = [(l + (' [COLAPSADA]' if COLAPSADO[l] else ''), e) for l, e in E4.items()]
tab_4c = R.table(ROWS, mode='4c', metrics=MET)
tab_4c.insert(len(MET), 'Grade', [g for *_, g, _ in PRINCIPAIS])
tab_sem_bg = R.table(ROWS, mode='4c_sem_bg', metrics=MET)
tab_3c = R.table([(l, e, '3c') for l, e in E3.items()], metrics=MET) if E3 else None
def _completo(e):
    try:
        R.fold_metrics(e)
        return True
    except FileNotFoundError:   # sem os 10 folds (ou CSVs com outro nome) em disco
        return False


SWEEP_FORA = [(l, e) for l, e in SVM_SWEEP if not _completo(e)]
SVM_SWEEP = [(l, e) for l, e in SVM_SWEEP if _completo(e)]
SWEEP = {l: R.summary(e) for l, e in SVM_SWEEP}
tab_sweep = R.table(SVM_SWEEP, metrics=MET)

print('Tabela 7 do artigo (4 classes):'); display(tab_artigo)
print('Avaliação local, 4 classes (modelos 4c):'); display(tab_4c)
print('Avaliação local, só navios, modelos 4c (prever BACKGROUND = erro):'); display(tab_sem_bg)
if tab_3c is not None:
    print('Só navios, modelos RETREINADOS sem BACKGROUND (3c):'); display(tab_3c)

### Leitura da Tabela Mestra
1. **Artigo:** SP com MEL na Tabela 7 — RF: 62,22 ± 1,86; MLP: 63,38 ± 1,81; CNN: 63,52 ± 2,26.
2. **Reprodução local do pipeline oficial:** .
3. **Ranking local (SP médio, 4c):** CNN (artigo) (65,53), MLP (≈ artigo) (64,74), MLP [32,16] 35 ép. (64,16), CNN (repo) (63,80), SVM Nyström m=4000 (63,78), BiGRU + atenção (63,60), Random Forest 100 árv. (61,50).

## 2. Comparativo Global

Índice SP e acurácia balanceada (média ± desvio nos 10 folds) no problema de 4 classes, com as linhas de referência da Tabela 7 do artigo para o MEL:

In [ ]:
lbl = list(E4)
fig, ax = plt.subplots(figsize=(15, 6))
x = np.arange(len(lbl)); w = 0.35
ax.bar(x - w / 2, [S4[l]['ACC'][0] for l in lbl], w, yerr=[S4[l]['ACC'][1] for l in lbl], capsize=3,
       label='ACC (acurácia balanceada)', color='#2980b9', alpha=0.9)
r = ax.bar(x + w / 2, [S4[l]['SP'][0] for l in lbl], w, yerr=[S4[l]['SP'][1] for l in lbl], capsize=3,
           label='SP', color='#e67e22', alpha=0.9)
for rect in r:
    ax.annotate(f'{rect.get_height():.1f}', xy=(rect.get_x() + rect.get_width() / 2, rect.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9, fontweight='bold')
for (m, rr), cor, ls in zip([k for k in R.ARTICLE_TABLE7 if k[1] == REP], ['#27ae60', '#8e44ad', '#7f8c8d'], ['--', ':', '-.']):
    v = R.ARTICLE_TABLE7[(m, rr)]['SP'][0]
    ax.axhline(v, color=cor, ls=ls, lw=1.8, label=f'{m} {REP}, Tabela 7 do artigo ({v:.2f}% SP)')
ax.set_xticks(x); ax.set_xticklabels([l + ('\n[COLAPSADA]' if COLAPSADO[l] else '') + f'\n({g})' for l, (*_, g, _) in zip(lbl, PRINCIPAIS)],
                                     fontweight='bold')
ax.set_ylabel('Média ± desvio nos 10 folds (%)'); ax.set_ylim(0, 80)
ax.legend(loc='lower left', fontsize=9, ncol=2, frameon=True)
plt.title(f'Modelos com o extrator {REP} — 4 classes (10 folds, 5x2cv oficial)', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Varredura do SVM Nyström

O SVM Nyström (núcleo RBF aproximado por *landmarks*, `test_scripts/svm.py`) foi a família mais explorada no projeto. A tabela e o gráfico abaixo trazem todas as configurações com MEL puro que têm os 10 folds em disco (cascatas e híbridos ficam no Notebook 5 e em `reports/comparacao_modelos.md`):

In [ ]:
lbls = [l for l, _ in SVM_SWEEP]
fig, ax = plt.subplots(figsize=(14, 5.5))
sp = [SWEEP[l]['SP'][0] for l in lbls]; sd = [SWEEP[l]['SP'][1] for l in lbls]
best_sw = int(np.argmax(sp))
bars = ax.bar(range(len(lbls)), sp, yerr=sd, capsize=3, color=['#c0392b' if i == best_sw else '#95a5a6' for i in range(len(lbls))])
for b in bars:
    ax.annotate(f'{b.get_height():.1f}', xy=(b.get_x() + b.get_width() / 2, b.get_height()), xytext=(0, 3),
                textcoords='offset points', ha='center', va='bottom', fontsize=8.5)
ax.set_xticks(range(len(lbls))); ax.set_xticklabels(lbls, rotation=30, ha='right')
ax.set_ylim(40, 75); ax.set_ylabel('SP (%) — média ± desvio nos 10 folds')
ax.set_title(f'Varredura do SVM Nyström com {REP} (pipeline oficial, janelas de 0,512 s)', fontweight='bold')
plt.tight_layout()
plt.show()

display(tab_sweep)

* A melhor configuração é **m=4000, C=2.0, PCA64, ElasticNet** (SP 63,82 ± 1,61%) e a pior, **m=1000, norm, PCA64** (SP 60,83 ± 1,67%). A diferença entre elas é significativa no F-test 5x2cv (F = 28,28, p < 0,001).
* 6 das outras 8 configurações ficam a menos de um desvio (1,61 p.p.) do SP médio da melhor; escolher uma "configuração campeã" só pelo SP médio não tem suporte estatístico sem o F-test.
* Ficaram fora da varredura, por não terem as predições dos 10 folds com o nome padrão em disco: m=300 (`svm_nystroem`).

## 4. Recall por Classe

Recall médio nos 10 folds (modelos 4c):

In [ ]:
classes = R.CLASSES
recalls_matrix = np.array([[S4[l][f'R_{c}'][0] for c in classes] for l in E4])
display(pd.DataFrame(recalls_matrix.round(2), index=list(E4), columns=classes))
fig, ax = plt.subplots(figsize=(14, 6))
n = len(E4); width = 0.8 / n
cores = ['#3498db', '#9b59b6', '#e67e22', '#e74c3c', '#16a085', '#7f8c8d']
for i, (l, color) in enumerate(zip(E4, cores)):
    rects = ax.bar(np.arange(4) + (i - (n - 1) / 2) * width, recalls_matrix[i], width, label=l, color=color, alpha=0.9)
    for rect in rects:
        ax.annotate(f'{rect.get_height():.0f}', xy=(rect.get_x() + rect.get_width() / 2, rect.get_height()),
                    xytext=(0, 2), textcoords='offset points', ha='center', va='bottom', fontsize=8)
ax.set_title(f'Recall por classe (média nos 10 folds) — 4 classes, {REP}', fontweight='bold', pad=15)
ax.set_xticks(np.arange(4)); ax.set_xticklabels(classes, fontsize=12, fontweight='bold')
ax.set_ylabel('Recall (%)'); ax.set_ylim(0, 100)
ax.legend(title='Modelo', loc='upper left', frameon=True, fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

### Principais Fenômenos Observados (valores calculados acima e nas matrizes da seção 5):
1. **SMALL:** o maior recall entre os modelos não colapsados é o do Random Forest 100 árv. (57,87%). Classe de menor recall em cada modelo: MLP (≈ artigo): SMALL (44,78%); CNN (artigo): SMALL (47,42%); CNN (repo): SMALL (40,90%); SVM Nyström m=4000: SMALL (50,73%); Random Forest 100 árv.: SMALL (57,87%); MLP [32,16] 35 ép.: SMALL (46,07%); BiGRU + atenção: SMALL (45,28%).
2. **BACKGROUND:** o maior recall é o do CNN (artigo) (88,47%); no CNN (artigo), 4,3% do fundo vai para SMALL.
3. **Confusão SMALL → MEDIUM / LARGE** (matrizes acumuladas): MLP (≈ artigo): 25,5% / 17,9%; CNN (artigo): 27,6% / 17,1%; CNN (repo): 34,2% / 15,6%; SVM Nyström m=4000: 24,3% / 17,7%; Random Forest 100 árv.: 21,7% / 18,5%; MLP [32,16] 35 ép.: 29,7% / 17,3%; BiGRU + atenção: 27,4% / 19,1%.

## 5. Matrizes de Confusão Acumuladas nos 10 Folds ($5\times2$cv)

### 5.1 Problema de 4 Classes (SMALL, MEDIUM, LARGE, BACKGROUND)
Matrizes por arquivo, **acumuladas** nas 10 dobras (n = 6.390 instâncias arquivo × fold; cada arquivo aparece em média 3,75 vezes no teste) e normalizadas por linha. A diagonal acumulada difere ligeiramente do recall médio por fold da tabela.

In [ ]:
def confusion(exp):
    """Matriz de confusão por arquivo, ACUMULADA nas 10 dobras (instâncias arquivo x fold), normalizada por linha (%)."""
    df = pd.concat(R.file_predictions(exp).values(), ignore_index=True)
    rows = sorted(df.Target.unique())
    cols = sorted(set(rows) | set(df.Prediction.unique()))
    cm = pd.crosstab(df.Target, df.Prediction).reindex(index=rows, columns=cols, fill_value=0)
    cmn = 100 * cm.div(cm.sum(axis=1), axis=0)
    cmn.index = [R.CLASSES[i] for i in rows]; cmn.columns = [R.CLASSES[i] for i in cols]
    return cmn, int(cm.values.sum())


CM4 = {l: confusion(e) for l, e in E4.items()}
CM3 = {l: confusion(e) for l, e in E3.items()}
N_FILES = len(set().union(*[set(d.File) for d in R.file_predictions(E4[next(iter(E4))]).values()]))


def plot_cms(CMS, S, cmap, suptitle):
    n = len(CMS); ncol = min(n, 3); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.2 * ncol, 5.6 * nrow), squeeze=False)
    for ax in axes.flatten()[n:]:
        ax.axis('off')
    for ax, (l, (cm, nn)) in zip(axes.flatten(), CMS.items()):
        sp = S[l]['SP']
        im = ax.imshow(cm.values, cmap=cmap, vmin=0, vmax=100)
        ax.set_title(f'{l}\nSP {sp[0]:.2f} ± {sp[1]:.2f} % (n = {nn})', fontsize=11, fontweight='bold')
        ax.set_xticks(range(cm.shape[1])); ax.set_xticklabels([c[:5] for c in cm.columns], fontweight='bold')
        ax.set_yticks(range(cm.shape[0])); ax.set_yticklabels([c[:5] for c in cm.index], fontweight='bold')
        ax.set_xlabel('Classe predita'); ax.set_ylabel('Classe real')
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                v = cm.values[i, j]
                ax.text(j, i, f'{v:.1f}%', ha='center', va='center', color='white' if v > 50 else 'black',
                        fontweight='bold' if i == j else 'normal', fontsize=9.5)
    plt.suptitle(suptitle, fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_cms(CM4, S4, 'Oranges', f'Matrizes de confusão normalizadas — {REP} [4 classes]')

### 5.2 Modelos Retreinados sem BACKGROUND (3c: SMALL, MEDIUM, LARGE)

Só MLP [32,16] 35 ép., BiGRU + atenção têm versão retreinada sem BACKGROUND com MEL. É outro problema (3 classes): os números não são comparáveis aos modelos 4c nem ao modo `4c_sem_bg`.

In [ ]:
plot_cms(CM3, S3, 'YlOrBr', f'Matrizes de confusão normalizadas — {REP} [3c, retreinado sem BACKGROUND]')

* **MLP [32,16] 35 ép.:** SP 3c 63,81 ± 1,81% (no 4c: 64,16 ± 2,19%); recall de SMALL 54,04% (no 4c: 46,07%); SMALL → LARGE 19,8% e SMALL → MEDIUM 26,2%.
* **BiGRU + atenção:** SP 3c 61,34 ± 2,49% (no 4c: 63,60 ± 1,53%); recall de SMALL 51,85% (no 4c: 45,28%); SMALL → LARGE 21,5% e SMALL → MEDIUM 26,7%.

## 6. Testes Estatísticos (F-test 5x2cv)

Toda afirmação de "supera" abaixo depende deste teste (Alpaydin, 1999; o mesmo usado no artigo). Primeiro, o melhor modelo MEL em SP médio contra os demais; depois, MEL × LOFAR no mesmo classificador:

In [ ]:
# F-test 5x2cv combinado (Alpaydin, 1999), como no artigo, sobre o SP por fold (modelos 4c)
ORDEM = sorted(E4, key=lambda l: -S4[l]['SP'][0])
MELHOR = ORDEM[0]
rows = []
for b in ORDEM[1:]:
    F, p = R.f_test_5x2(E4[MELHOR], E4[b])
    rows.append({'A': MELHOR, 'B': b, 'SP A': round(S4[MELHOR]['SP'][0], 2), 'SP B': round(S4[b]['SP'][0], 2),
                 'F': round(F, 2), 'p': round(p, 4), 'p < 0,05': p < 0.05})
df_tests = pd.DataFrame(rows)
print(f'Melhor modelo {REP} (SP médio) contra os demais:')
display(df_tests)

rows = []
for arq, a, b in PARES_OUTRA:
    sa, sb = R.summary(a), R.summary(b)
    F, p = R.f_test_5x2(a, b)
    Fs, ps = R.f_test_5x2(a, b, metric='R_SMALL')
    rows.append({'Arquitetura': arq, f'SP {REP}': R.fmt(sa['SP']), f'SP {OUTRA}': R.fmt(sb['SP']),
                 'Δ SP (p.p.)': round(sa['SP'][0] - sb['SP'][0], 2), 'p (SP)': round(p, 4),
                 f'R_SMALL {REP}': round(sa['R_SMALL'][0], 2), f'R_SMALL {OUTRA}': round(sb['R_SMALL'][0], 2),
                 'p (R_SMALL)': round(ps, 4), 'exp A': a, 'exp B': b})
df_outra = pd.DataFrame(rows).set_index('Arquitetura')
print(f'{REP} × {OUTRA} no mesmo classificador:')
display(df_outra.drop(columns=['exp A', 'exp B']))

## 7. Diagnóstico Científico e Conclusões

### 7.1 O melhor modelo MEL
* Em SP médio, o melhor é **CNN (artigo)** (65,53 ± 1,73%), que é significativamente melhor que CNN (repo) e equivalente a MLP (≈ artigo), MLP [32,16] 35 ép., SVM Nyström m=4000, BiGRU + atenção, Random Forest 100 árv..
* A maior diferença de SP médio **sem** significância é de 4,03 p.p. (CNN (artigo) × Random Forest 100 árv.): a ordem do ranking da seção 1 **não** deve ser lida como hierarquia.

* Referência do projeto: a MLP LOFAR (artigo 66,51 ± 1,39%; local 66,24 ± 1,78%). O melhor MEL contra a MLP LOFAR local: F-test p = 0,674 (**não** significativa).

### 7.2 MEL × LOFAR no mesmo classificador

| Arquitetura | SP MEL | SP LOFAR | Δ (p.p.) | p (SP) | R_SMALL MEL | R_SMALL LOFAR | p (R_SMALL) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| MLP (≈ artigo, 0,512 s) | 64,74 ± 2,05 | 66,24 ± 1,78 | -1,51 | 0,653 | 44,78 | 46,35 | 0,489 |
| SVM Nyström (melhor de cada) | 63,78 ± 1,14 | 63,83 ± 1,91 | -0,05 | 0,096 | 50,73 | 38,15 | 0,263 |
| Random Forest (5,12 s) | 61,50 ± 1,89 | 63,07 ± 1,81 | -1,57 | 0,637 | 57,87 | 47,81 | 0,032 |
| BiGRU + atenção (5,12 s) | 63,60 ± 1,53 | 62,22 ± 1,09 | 1,38 | 0,214 | 45,28 | 55,84 | 0,235 |

* A diferença de SP entre MEL e LOFAR é significativa (p < 0,05) em: **nenhuma** arquitetura.
* No recall de SMALL, a diferença é significativa em: Random Forest (5,12 s) (MEL 57,87% × LOFAR 47,81%, p = 0,032).
* No artigo, a ordem entre LOFAR e MEL também depende do classificador (Tabela 7: RF favorece o MEL; MLP e CNN favorecem o LOFAR).

### 7.3 O que estes resultados permitem dizer
1. Os números do pipeline oficial reproduzem os do artigo dentro do desvio entre folds (seção 1), o que dá base para as comparações locais.
2. Os modelos ad hoc de 5,12 s não são uma reprodução do artigo: mudam a grade temporal **e** a implementação; servem para comparar representações (LOFAR, MEL, DEMON, HHT) no mesmo classificador, como nos Notebooks 9, 11 e 12.
3. Afirmações de superioridade entre modelos MEL só valem quando o F-test da seção 6 dá p < 0,05.